# Manual Bidding helper screen

In [1]:
# Get daily constraint ranked by the abs RT - DA 
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')

import pandas as pd
from nighthawk.data import Constraint

In [2]:
now = pd.Timestamp.now(tz='US/Central')
days_ahead = 2 if now.hour >= 10 else 1
bid_dt = (now + pd.Timedelta(days=days_ahead)).strftime('%Y-%m-%d')
print(bid_dt)

2026-04-28


In [9]:

opex = 'SPP'
end_dt = pd.Timestamp.now(tz='US/Central').strftime('%Y-%m-%d')
start_dt = (pd.Timestamp(end_dt) - pd.Timedelta(days=3)).strftime('%Y-%m-%d')

end_dt = '2026-04-01'
start_dt = '2025-04-01'

print(f"start_dt: {start_dt}, end_dt: {end_dt}, Market: {opex}")

# Get RT and DA hourly mvalues for all constraints over the date range
print("Fetching RT mvalues...")
mv_rt = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
    start_dt=start_dt, end_dt=end_dt, type='RT', granularity='hourly')

print("Fetching DA mvalues...")
mv_da = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
    start_dt=start_dt, end_dt=end_dt, type='DA', granularity='hourly')

print(f"RT constraints: {mv_rt['oops_constraint_num'].nunique()}, DA constraints: {mv_da['oops_constraint_num'].nunique()}")

# Sum mvalues per constraint
rt_total = mv_rt.groupby('oops_constraint_num')['mvalue'].sum().rename('rt_total')
da_total = mv_da.groupby('oops_constraint_num')['mvalue'].sum().rename('da_total')

# Merge and compute abs(RT - DA), ranked descending
merged = pd.merge(rt_total, da_total, on='oops_constraint_num', how='outer').fillna(0)
merged['rt_da'] = (merged['rt_total'] - merged['da_total'])
merged['abs_rt_da_diff'] = (merged['rt_total'] - merged['da_total']).abs()
merged = merged.sort_values('abs_rt_da_diff', ascending=False).reset_index()

# Get constraint details
all_cons = merged[['oops_constraint_num']].copy()
print("Fetching constraint details...")
details_rt = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='RT')
details_da = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='DA')
details = pd.concat([details_rt, details_da]).drop_duplicates('oops_constraint_num')

# Get linked outages for each date in the range and concat
print("Fetching linked outages for each date...")
outage_frames = []
for dt in pd.date_range(start_dt, end_dt, freq='D'):
    dt_str = dt.strftime('%Y-%m-%d')
    df_day = Constraint(oops_constraint_num_df=all_cons, market=opex).get_linked_outage_string_for_frontend(dt=dt_str)
    df_day['dt'] = dt_str
    outage_frames.append(df_day)

# Concat and keep last (most recent) outage string per constraint
linked_outages = (
    pd.concat(outage_frames, ignore_index=True)
    .dropna(subset=['linkedOutage'])
    .drop_duplicates(subset='oops_constraint_num', keep='last')
    [['oops_constraint_num', 'linkedOutage']]
)

# Merge everything
result = pd.merge(merged, details[['oops_constraint_num', 'monitored_clean', 'contingency_clean']], on='oops_constraint_num', how='left')
result = pd.merge(result, linked_outages, on='oops_constraint_num', how='left')
result = result.rename(columns={'monitored_clean': 'monitored', 'contingency_clean': 'contingency'})
result.drop_duplicates(subset=['monitored'], inplace=True)
result = result[result['abs_rt_da_diff']>3000]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)
display(result[['oops_constraint_num', 'rt_total', 'da_total', 'rt_da', 'abs_rt_da_diff', 'monitored', 'contingency', 'linkedOutage']])


start_dt: 2025-04-01, end_dt: 2026-04-01, Market: SPP
Fetching RT mvalues...
Fetching DA mvalues...
RT constraints: 1056, DA constraints: 2074
Fetching constraint details...
Fetching linked outages for each date...


KeyboardInterrupt: 

In [4]:
# Get Outage information associated with each constraint

import sys
sys.path.append('/var/www/python/Prod/nighthawk/')
import pandas as pd
from nighthawk.data.network.outage import Outage
from nighthawk.data.network.powerflow import Powerflow
from nighthawk.data import Constraint

# Fetch all planned + actual outages for the date range
viewing_time = end_dt + " 01:00:00"
outage_obj = Outage(opex)

print("Fetching planned outages...")
da = outage_obj.get_scheduled_outages_for_frontend(start_dt, end_dt, viewing_time)
print("Fetching actual outages...")
rt = outage_obj.get_realtime_outages_for_frontend(start_dt, end_dt, viewing_time)

da = da[~da['eqNum'].isna()]
rt = rt[~rt['eqNum'].isna()]

# Merge DA and RT into one combined outage table
combined_outages = pd.merge(
    da.rename(columns={'outageStartDtTime': 'planned_start_dt', 'outageEndDtTime': 'planned_end_dt'}),
    rt.rename(columns={'outageStartDtTime': 'actual_start_dt', 'outageEndDtTime': 'actual_end_dt'}),
    on=['eqNum'], how='outer'
)
combined_outages['outageName'] = combined_outages['outageName_x'].combine_first(combined_outages['outageName_y'])
combined_outages['KV'] = combined_outages['voltage_x'].combine_first(combined_outages['voltage_y'])
combined_outages['outageNum'] = combined_outages['outageNum_x'].combine_first(combined_outages['outageNum_y'])
combined_outages = combined_outages[
    ['eqNum', 'outageNum', 'outageName', 'KV', 'planned_start_dt', 'planned_end_dt', 'actual_start_dt', 'actual_end_dt']
].copy()
for col in ['planned_start_dt', 'planned_end_dt', 'actual_start_dt', 'actual_end_dt']:
    combined_outages[col] = pd.to_datetime(combined_outages[col], errors='coerce').dt.strftime("%Y-%m-%d %H:%M")
print(f"Total combined outages: {len(combined_outages)}")

# Get monitored eqNum for each constraint in the top 30 result
top_cons = result.head(30)[['oops_constraint_num']].copy()
print("Fetching monitored eqNums...")
monitored_eq = Constraint(oops_constraint_num_df=top_cons, market=opex).get_monitored_equipment_details()
monitored_eq = monitored_eq[['oops_constraint_num', 'monitored_eqNum']].drop_duplicates()
print(f"Constraints with monitored eqNum: {len(monitored_eq)}")

# Build cross product of all (monitored eqNum, outage eqNum) pairs
outage_eqnums = combined_outages[['eqNum']].drop_duplicates().rename(columns={'eqNum': 'eqNum2'})
mon_eqnums = monitored_eq[['monitored_eqNum']].drop_duplicates().rename(columns={'monitored_eqNum': 'eqNum1'})
eq_pairs = mon_eqnums.merge(outage_eqnums, how='cross')
print(f"Total (monitored, outage) pairs: {len(eq_pairs)}")

# Get powerflow model and compute LODF/perChange + distance
print("Getting powerflow model...")
model_num = Powerflow(opex).get_latest_model_num_for_dt(end_dt)
pf = Powerflow(opex, model_num=model_num)

print("Computing LODF/perChange...")
lodf_df = pf.get_lodf_perchange_info(eq_pairs)
print("Computing distance...")
dist_df = pf.get_distance_between_equipments(eq_pairs)

# Merge LODF and distance metrics
metrics = pd.merge(lodf_df, dist_df, on=['eqNum1', 'eqNum2'], how='outer')

# Join back with outage details and constraint mapping
outage_metrics = (
    metrics
    .rename(columns={'eqNum1': 'monitored_eqNum', 'eqNum2': 'eqNum'})
    .merge(monitored_eq, on='monitored_eqNum', how='left')
    .merge(combined_outages, on='eqNum', how='left')
)

# Join with constraint summary (monitored name, contingency, abs diff for ordering)
outage_result = outage_metrics.merge(
    result.head(30)[['oops_constraint_num', 'monitored', 'contingency', 'abs_rt_da_diff']],
    on='oops_constraint_num', how='left'
)
outage_result = outage_result.sort_values(['abs_rt_da_diff', 'perChange'], ascending=[False, False]).reset_index(drop=True)

display_cols = ['oops_constraint_num', 'monitored', 'contingency', 'outageName', 'KV',
                'planned_start_dt', 'planned_end_dt', 'actual_start_dt', 'actual_end_dt',
                'LODF', 'perChange', 'distance']

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 320)
pd.set_option('display.max_colwidth', 60)
display(outage_result[display_cols])


Fetching planned outages...
Fetching actual outages...


/tmp/ipykernel_422048/498174352.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_outages[col] = pd.to_datetime(combined_outages[col], errors='coerce').dt.strftime("%Y-%m-%d %H:%M")


Total combined outages: 476
Fetching monitored eqNums...
Constraints with monitored eqNum: 17
Total (monitored, outage) pairs: 5729
Getting powerflow model...
Computing LODF/perChange...
Computing distance...


,oops_constraint_num,monitored,contingency,outageName,KV,planned_start_dt,planned_end_dt,actual_start_dt,actual_end_dt,LODF,perChange,distance
0,649753,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34,Brown SPA - Tupelo 138 kV,138.0,2026-04-27 10:00,2026-04-30 16:00,NaN,NaN,-41.37900,11.710123,0.0
1,649753,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34,Canadian River 345/138 kV Transformer,138.0,2026-04-17 08:00,2026-05-08 16:00,NaN,NaN,-3.07208,8.989643,NaN
2,649753,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34,Johnston County - Sunnyside 345kV,345.0,2026-01-09 08:00,2026-05-20 16:00,2026-01-09 08:30,NaN,-2.72475,5.120392,4.0
3,649753,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34,Loco - Pinto 138 kV,138.0,2026-04-20 08:00,2026-04-24 16:00,NaN,NaN,-2.50463,1.984229,NaN
4,649753,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34,Loco - Pinto 138 kV,138.0,2026-04-27 08:00,2026-04-30 16:01,NaN,NaN,-2.50463,1.984229,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
361,315182,lncolby3-atwood,seci:setab1mingo1:345:2:13,Ionia Substation - Jewell 115 kV,115.0,2026-03-30 08:00,2026-04-24 18:00,NaN,NaN,-3.69612,1.111460,NaN
362,315182,lncolby3-atwood,seci:setab1mingo1:345:2:13,Ionia Substation - Smith Center 115 kV,115.0,2026-03-30 08:00,2026-04-24 18:00,NaN,NaN,3.69612,1.069506,NaN
363,315182,lncolby3-atwood,seci:setab1mingo1:345:2:13,Covert Substation - Smith Center 115 kV,115.0,2026-04-27 08:00,2026-04-28 17:00,NaN,NaN,-4.37422,-1.358743,NaN
364,315182,lncolby3-atwood,seci:setab1mingo1:345:2:13,Gentleman - North Platte 230 kV #2,230.0,2026-03-21 08:00,2026-04-30 16:00,2026-03-21 08:43,NaN,NaN,NaN,6.0


In [5]:
# Filter the important outage information 

temp = outage_result[outage_result['actual_end_dt'].astype(str).str[:10] >= bid_dt ]
temp2 = temp[temp['planned_end_dt'].astype(str).str[:10] >= bid_dt]


In [6]:
temp3 = temp2[(temp2['distance'].notnull()) & (temp2['distance']<=3)]
temp4 = temp3[abs(temp3['LODF']>3)]

In [7]:
temp4.monitored.unique()

array(['multi-elementconstraint.tmp800_29528.tmp800_29528',
       'lnwolffort-terry_co', 'xfmrduncan-duncan'], dtype=object)

In [8]:
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')
import pandas as pd
from nighthawk.data import Constraint
from nighthawk.data.network.constraint_family import ConstraintFamily

opex = 'SPP'
oops_constraint_num = 874086  # your constraint num

# Step 1: get constraint family num
cons_df = pd.DataFrame({'oops_constraint_num': [oops_constraint_num]})
cfn = Constraint(oops_constraint_num_df=cons_df, market=opex).get_constraint_family_num()
constraint_family_num = cfn['constraint_family_num'].iloc[0]
print(f"constraint_family_num: {constraint_family_num}")

# Step 2: pull historical daily mvalues
cfn_df = pd.DataFrame({'constraintFamilyNum': [constraint_family_num]})
cf = ConstraintFamily(opexchange=opex, constraint_family_num_df=cfn_df)

history = cf.get_mvalues(
    start_dt='2025-01-01',
    end_dt=end_dt,
    type='RT',         # or 'RT'
    granularity='daily'  # or 'hourly'
)
print(history.shape)
display(history.sort_values('dt', ascending=False).head(30))

constraint_family_num: 881
(10, 3)


,constraintFamilyNum,dt,mvalue
9,881,2026-04-26,-1326.6929
8,881,2026-04-25,-1137.2866
7,881,2026-04-24,-1084.0164
6,881,2026-04-23,-25.2262
5,881,2026-02-19,-39.1402
4,881,2026-01-24,-29.9755
3,881,2025-03-25,-1426.7047
2,881,2025-03-24,-857.5709
1,881,2025-03-23,-6024.6944
0,881,2025-03-22,-3260.5191
